In [ ]:
#@title 0. Instalar dependencias y montar Drive
from google.colab import drive
drive.mount('/content/drive')

# Instalar detectron2 desde fuente
import sys, os, distutils.core

# Detectron2 (fast local install)
!git clone -q 'https://github.com/facebookresearch/detectron2'
dist = distutils.core.run_setup("./detectron2/setup.py")
!python -m pip install -q {' '.join([f"'{x}'" for x in dist.install_requires])}
import sys, os
sys.path.insert(0, os.path.abspath('./detectron2'))

# Otros paquetes
!pip install -q rasterio pyproj fiona matplotlib albumentations tqdm pycocotools pandas scikit-image

In [ ]:
#@title 1. Rutas, constantes y utilidades de tamaños
from pathlib import Path
import os, math, json
import numpy as np
import pandas as pd

BASE = Path("/content/drive/MyDrive/juniper_mapper/JuniperMapper")

# COCO instancia (para evaluar IoU/S-IoU por instancia)
PI_COCO_JSON = str(BASE/"Photo_Interpretation_Data/Test/Annotations/Test.json")
PI_IMG_DIR   = str(BASE/"Photo_Interpretation_Data/Test/Images")
FW_COCO_JSON = str(BASE/"Field_Work_Data/External_Val_Data/Annotations/FW.json")
FW_IMG_DIR   = str(BASE/"Field_Work_Data/External_Val_Data/Images")

# Máscaras GT semánticas (binarias 0/1)
GT_PI = str(BASE/"Photo_Interpretation_Data/Test/Annotations/Masks")
GT_FW = str(BASE/"Field_Work_Data/External_Val_Data/Annotations/Masks")

# Salidas DeepLab
OUT_PI  = str(BASE/"DeepLab/output_PI")
OUT_FW  = str(BASE/"DeepLab/output_FW")
PRED_PI = f"{OUT_PI}/Predictions"
PRED_FW = f"{OUT_FW}/Predictions"

for d in [OUT_PI, OUT_FW, PRED_PI, PRED_FW, f"{OUT_PI}/csv", f"{OUT_PI}/tables"]:
    os.makedirs(d, exist_ok=True)

# Bins de tamaño (m²)
SIZE_BINS = [
    ("XS",  0.13,  1.72),
    ("S",   1.72,  3.62),
    ("M",   3.62,  9.08),
    ("L",   9.08, 20.82),
    ("XL", 20.82, 41.06),
    ("XXL",41.06, float("inf")),
]
def size_label(area_m2: float):
    for name, lo, hi in SIZE_BINS:
        if lo <= area_m2 < hi:
            return name
    return "XS"

# Semillas
SEED = 1337

In [ ]:
#@title 2. Registro datasets semánticos (PI/FW)
import rasterio
from detectron2.data import DatasetCatalog, MetadataCatalog

def cargar_dataset_semantico(im_dir, mask_dir):
    import os
    dataset_dicts = []
    for filename in sorted(os.listdir(im_dir)):
        if not filename.endswith(".tif"):
            continue
        image_id = filename[:-4]  # "Img_123"
        img_path = os.path.join(im_dir, filename)
        # GT: "Mask_123.tif"
        mask_idx = image_id.split('_')[-1]
        mask_path = os.path.join(mask_dir, f"Mask_{mask_idx}.tif")
        with rasterio.open(img_path) as src:
            height, width = src.height, src.width
        dataset_dicts.append({
            "file_name": img_path,
            "image_id": image_id,
            "sem_seg_file_name": mask_path,
            "height": height,
            "width": width,
        })
    return dataset_dicts

def registrar_dataset(nombre, im_dir, mask_dir):
    DatasetCatalog.register(nombre, lambda: cargar_dataset_semantico(im_dir, mask_dir))
    MetadataCatalog.get(nombre).set(
        stuff_classes=["fondo", "juniperus"],
        evaluator_type="sem_seg",
        ignore_label=255
    )

registrar_dataset("pi_train_sem", str(BASE/"Photo_Interpretation_Data/Train/Images"), str(BASE/"Photo_Interpretation_Data/Train/Annotations/Masks"))
registrar_dataset("pi_val_sem",   str(BASE/"Photo_Interpretation_Data/Val/Images"),   str(BASE/"Photo_Interpretation_Data/Val/Annotations/Masks"))
registrar_dataset("pi_test_sem",  PI_IMG_DIR, GT_PI)
registrar_dataset("fw_test_sem",  FW_IMG_DIR, GT_FW)
PI_TRAIN="pi_train_sem"; PI_VAL="pi_val_sem"; PI_TEST="pi_test_sem"; FW_TEST="fw_test_sem"

In [ ]:
#@title 3. Configuración + Entrenamiento DeepLabV3+
import os, random, numpy as np, torch, torch.nn as nn
from detectron2.config import get_cfg
from detectron2.projects.deeplab import add_deeplab_config, build_lr_scheduler as build_deeplab_lr_scheduler
from detectron2.engine import DefaultTrainer
from detectron2.modeling import build_model

# YAML base
DEEPLAB_CFG = "/content/detectron2/projects/DeepLab/configs/Cityscapes-SemanticSegmentation/deeplab_v3_plus_R_103_os16_mg124_poly_90k_bs16.yaml"
cfg = get_cfg(); add_deeplab_config(cfg)
cfg.merge_from_file(DEEPLAB_CFG)

# Datasets
cfg.DATASETS.TRAIN = (PI_TRAIN,)
cfg.DATASETS.TEST  = (PI_VAL,)

# Modelo
cfg.MODEL.SEM_SEG_HEAD.NUM_CLASSES  = 2
cfg.MODEL.SEM_SEG_HEAD.IGNORE_VALUE = 255
cfg.MODEL.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Tamaños (422→448, múltiplo de 32)
PI_TILE = 448
cfg.INPUT.CROP.ENABLED   = True
cfg.INPUT.CROP.TYPE      = "absolute"
cfg.INPUT.CROP.SIZE_TRAIN= (PI_TILE, PI_TILE)
cfg.INPUT.MIN_SIZE_TRAIN = (PI_TILE,)
cfg.INPUT.MAX_SIZE_TRAIN = PI_TILE
cfg.INPUT.MIN_SIZE_TEST  = PI_TILE
cfg.INPUT.MAX_SIZE_TEST  = PI_TILE
cfg.INPUT.SIZE_DIVISIBILITY = 32
cfg.INPUT.RANDOM_FLIP    = "horizontal"
cfg.MODEL.SEM_SEG_HEAD.TRAIN_SIZE = (PI_TILE, PI_TILE)

# Salidas
cfg.OUTPUT_DIR = "/content/drive/MyDrive/juniper_mapper/JuniperMapper/DeepLabV3/output_PI"
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

# Parche ASPP para robustez
def patch_aspp_fully(model: torch.nn.Module):
    aspp_attr_fixed = 0
    avg_replaced = 0
    for mod in model.modules():
        if hasattr(mod, "pool_kernel_size"):
            mod.pool_kernel_size = None
            aspp_attr_fixed += 1
    for name, module in model.named_modules():
        if isinstance(module, nn.Sequential) and len(module) > 0 and isinstance(module[0], nn.AvgPool2d):
            module[0] = nn.AdaptiveAvgPool2d(1)
            avg_replaced += 1
    print(f"ASPP parcheado: pool_kernel_size=None en {aspp_attr_fixed} módulos; AvgPool2d→AdaptiveAvgPool2d(1) en {avg_replaced} ramas.")

class DeepLabTrainer(DefaultTrainer):
    @classmethod
    def build_model(cls, cfg):
        model = build_model(cfg)
        patch_aspp_fully(model)
        return model
    @classmethod
    def build_lr_scheduler(cls, cfg, optimizer):
        return build_deeplab_lr_scheduler(cfg, optimizer)

# Semillas deterministas
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# Optimizador / LR
cfg.SOLVER.MAX_ITER = 12000
cfg.SOLVER.BASE_LR  = 2e-4
cfg.SOLVER.WARMUP_ITERS = 500
cfg.SOLVER.CHECKPOINT_PERIOD = 2000
cfg.SOLVER.AMP.ENABLED       = True
cfg.DATALOADER.NUM_WORKERS   = 2

# Entrenamiento
DO_TRAIN = False
if DO_TRAIN:
    trainer = DeepLabTrainer(cfg)
    trainer.resume_or_load(resume=False)
    trainer.train()
final_ckpt = os.path.join(cfg.OUTPUT_DIR, "model_final.pth")
if os.path.exists(final_ckpt):
    cfg.MODEL.WEIGHTS = final_ckpt

In [ ]:
#@title 4. Predictors PI/FW + parche ASPP
import cv2, copy, torch
from detectron2.engine import DefaultPredictor
import torch.nn as nn

PI_TILE_TEST = 448

def _patch_aspp_inference(model: torch.nn.Module):
    for mod in model.modules():
        if hasattr(mod, "pool_kernel_size"):
            mod.pool_kernel_size = None
    for name, module in model.named_modules():
        if isinstance(module, nn.Sequential) and len(module) > 0 and isinstance(module[0], nn.AvgPool2d):
            module[0] = nn.AdaptiveAvgPool2d(1)

class PreResizePredictor:
    def __init__(self, base_predictor, size_hw: int):
        self.base = base_predictor
        self.size = int(size_hw)
        _patch_aspp_inference(self.base.model)
    def __call__(self, img_bgr):
        resized = cv2.resize(img_bgr, (self.size, self.size), interpolation=cv2.INTER_LINEAR)
        return self.base(resized)

# Predictor PI
predictor_pi_core = DefaultPredictor(cfg)
predictor_pi      = PreResizePredictor(predictor_pi_core, PI_TILE_TEST)

# Predictor FW
cfg_fw = copy.deepcopy(cfg)
cfg_fw.DATASETS.TEST   = (FW_TEST,)
cfg_fw.INPUT.MIN_SIZE_TEST = PI_TILE
cfg_fw.INPUT.MAX_SIZE_TEST = PI_TILE
predictor_fw_core = DefaultPredictor(cfg_fw)
predictor_fw      = PreResizePredictor(predictor_fw_core, PI_TILE_TEST)

def softmax_numpy(logits):
    x = logits - logits.max(axis=0, keepdims=True)
    e = np.exp(x)
    return e / (e.sum(axis=0, keepdims=True) + 1e-12)

@torch.no_grad()
def _logits_from_predictor(pred, img_bgr):
    """
    Devuelve logits (C,H,W) de una única pasada por el predictor.
    """
    out = pred(img_bgr)
    log = out["sem_seg"]
    return log.detach().cpu().numpy() if hasattr(log, "detach") else log

def _resize_logits_to(logits, H, W):
    C, h, w = logits.shape
    return np.stack(
        [cv2.resize(logits[c], (W, H), interpolation=cv2.INTER_LINEAR) for c in range(C)],
        axis=0
    )

def tta_probs(pred, img_bgr):
    """
    Devuelve probabilidades por clase (C,H,W) usando una sola predicción (sin TTA).
    """
    H, W = img_bgr.shape[:2]
    logits = _logits_from_predictor(pred, img_bgr)
    logits_resized = _resize_logits_to(logits, H, W)
    return softmax_numpy(logits_resized)

In [ ]:
#@title 5. SemSeg→Instancias y métricas
import cv2, rasterio
from scipy import ndimage
from pycocotools.coco import COCO
from scipy.optimize import linear_sum_assignment

def image_gsd_m(geotiff_path, fallback=0.13):
    try:
        with rasterio.open(geotiff_path) as src:
            tr = src.transform
            if src.crs and src.crs.is_projected:
                px_m = abs(tr.a); py_m = abs(tr.e)
            else:
                mx = 111320.0; my = 110540.0
                px_m = mx * abs(tr.a); py_m = my * abs(tr.e)
            if px_m > 0 and py_m > 0:
                return float((px_m + py_m) / 2.0)
    except Exception:
        pass
    return float(fallback)

def mask_area_m2(mask_bool, gsd_m):
    return float(mask_bool.sum()) * (gsd_m ** 2)

def ann_masks_for_img(coco: COCO, img_id: int, H: int, W: int):
    ann_ids = coco.getAnnIds(imgIds=[img_id])
    anns = coco.loadAnns(ann_ids)
    masks = []
    for a in anns:
        m = coco.annToMask(a).astype(bool)
        if m.shape != (H, W):
            m = cv2.resize(m.astype(np.uint8), (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)
        masks.append(m)
    return masks

def semseg_to_instances_with_morph(
    prob_map, thr, gsd_m, min_area_m2=0.13, morph_close_ks=0, connectivity=2,
    min_score=None, morph_open_ks=0, smooth_sigma=0.0, sizeaware_min_score=None, sizeaware_cut_m2=3.62
):
    pm_prob = prob_map
    if smooth_sigma and smooth_sigma > 0:
        pm_prob = cv2.GaussianBlur(pm_prob, (0, 0), float(smooth_sigma))
    pm = (pm_prob >= float(thr)).astype(np.uint8)

    if morph_open_ks and morph_open_ks > 0:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (int(morph_open_ks), int(morph_open_ks)))
        pm = cv2.morphologyEx(pm, cv2.MORPH_OPEN, k)
    if morph_close_ks and morph_close_ks > 0:
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (int(morph_close_ks), int(morph_close_ks)))
        pm = cv2.morphologyEx(pm, cv2.MORPH_CLOSE, k)

    lab, nlab = ndimage.label(pm, structure=ndimage.generate_binary_structure(2, connectivity))
    min_area_px = (float(min_area_m2) / (gsd_m**2))
    pred_masks, pred_scores = [], []
    for kidx in range(1, nlab + 1):
        comp = (lab == kidx)
        area_px = comp.sum()
        if area_px < min_area_px:
            continue
        score = float(pm_prob[comp].mean())  # score de instancia
        if (min_score is not None) and (score < float(min_score)):
            continue
        if sizeaware_min_score is not None:
            area_m2 = area_px * (gsd_m**2)
            if area_m2 < float(sizeaware_cut_m2) and score < float(sizeaware_min_score):
                continue
        pred_masks.append(comp.astype(bool))
        pred_scores.append(score)
    return pred_masks, pred_scores, (pm.astype(bool))

def pixel_metrics(pred_binary: np.ndarray, gt_binary: np.ndarray):
    y_pred = pred_binary.astype(np.uint8).ravel()
    y_true = gt_binary.astype(np.uint8).ravel()
    tp = int(np.sum((y_true==1) & (y_pred==1)))
    tn = int(np.sum((y_true==0) & (y_pred==0)))
    fp = int(np.sum((y_true==0) & (y_pred==1)))
    fn = int(np.sum((y_true==1) & (y_pred==0)))
    total = tp + tn + fp + fn
    acc = (tp + tn) / total if total > 0 else 0.0
    iou_fg = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0
    iou_bg = tn / (tn + fp + fn) if (tn + fp + fn) > 0 else 0.0
    miou = 0.5 * (iou_fg + iou_bg)
    w0 = (tn + fp) / total if total > 0 else 0.0
    w1 = (tp + fn) / total if total > 0 else 0.0
    fwIoU = w0 * iou_bg + w1 * iou_fg
    return dict(pACC=acc, mIoU=miou, fwIoU=fwIoU)

def iou_matrix(pred_masks, gt_masks):
    if len(pred_masks)==0 or len(gt_masks)==0:
        return np.zeros((len(pred_masks), len(gt_masks)), dtype=float)
    P, G = len(pred_masks), len(gt_masks)
    M = np.zeros((P, G), dtype=float)
    for i, p in enumerate(pred_masks):
        ip = p.astype(bool); p_area = ip.sum()
        for j, g in enumerate(gt_masks):
            ig = g.astype(bool)
            inter = np.logical_and(ip, ig).sum()
            union = p_area + ig.sum() - inter
            M[i, j] = float(inter / max(1, union))
    return M

def eval_iou_at_threshold_hungarian(pred_masks, gt_masks, iou_thr=0.5):
    iou = iou_matrix(pred_masks, gt_masks)
    if iou.size == 0:
        return dict(tp=0, fp=len(pred_masks), fn=len(gt_masks))
    cost = 1.0 - iou
    ri, cj = linear_sum_assignment(cost)
    matched_pred = set(); matched_gt = set()
    for i, j in zip(ri, cj):
        if iou[i, j] >= iou_thr:
            matched_pred.add(i); matched_gt.add(j)
    tp = len(matched_pred)
    fp = len(pred_masks) - tp
    fn = len(gt_masks) - tp
    return dict(tp=tp, fp=fp, fn=fn)

def siou_pred(p_mask: np.ndarray, gt_masks):
    matches = [g for g in gt_masks if np.any(p_mask & g)]
    if not matches:
        return 0.0
    union_gt = np.any(np.stack(matches, axis=0), axis=0)
    inter = np.logical_and(p_mask, union_gt).sum()
    den = union_gt.sum()
    return float(inter / den) if den > 0 else 0.0

def siou_label(l_mask: np.ndarray, pred_masks):
    matches = [p for p in pred_masks if np.any(l_mask & p)]
    if not matches:
        return 0.0
    union_pr = np.any(np.stack(matches, axis=0), axis=0)
    inter = np.logical_and(l_mask, union_pr).sum()
    den = l_mask.sum()
    return float(inter / den) if den > 0 else 0.0

def eval_siou_at_threshold(pred_masks, pred_scores, gt_masks, siou_thr=0.5):
    order = np.argsort(-np.asarray(pred_scores)) if len(pred_scores)>0 else np.arange(len(pred_masks))
    tp = fp = 0
    for i in order:
        s = siou_pred(pred_masks[i], gt_masks)
        if s >= siou_thr:
            tp += 1
        else:
            fp += 1
    fn = 0
    for g in gt_masks:
        s = siou_label(g, pred_masks)
        if s < siou_thr:
            fn += 1
    prec = tp/(tp+fp) if (tp+fp)>0 else 0.0
    rec  = tp/(tp+fn) if (tp+fn)>0 else 0.0
    f1   = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0.0
    return dict(tp=tp, fp=fp, fn=fn, precision=prec, recall=rec, f1=f1)

In [ ]:
#@title 6. Evaluador (curvas F1 vs θ, θ*, tablas, etc.) + guardado de máscaras a θ*
import matplotlib.pyplot as plt
from tqdm import tqdm

def evaluate_semseg_vs_coco_params_hungarian(
    coco_json, img_dir, predictor, inst_params,
    iou_thrs=(0.5,0.75), siou_thrs=(0.5,0.75),
    write_pred_masks_dir=None, model_tag="DeepLab",
    base_pixel_thr=None, theta_grid=np.linspace(0.05, 0.95, 19)
):
    thr_components = float(inst_params.get("th", 0.50)) if base_pixel_thr is None else float(base_pixel_thr)
    morph_open_ks  = int(inst_params.get("morph_open_ks", 0))
    morph_close_ks = int(inst_params.get("morph_close_ks", 0))
    min_area_px    = int(inst_params.get("min_area_px", 0))
    min_score_base = inst_params.get("min_score", None)  # None => se barre θ

    coco = COCO(coco_json); images = coco.loadImgs(coco.getImgIds())

    res_counts = {("IoU",t): dict(tp=0,fp=0,fn=0) for t in iou_thrs}
    res_counts.update({("S-IoU",t): dict(tp=0,fp=0,fn=0) for t in siou_thrs})
    curves_counts = {("IoU",t): [dict(tp=0,fp=0,fn=0) for _ in theta_grid] for t in iou_thrs}
    curves_counts.update({("S-IoU",t): [dict(tp=0,fp=0,fn=0) for _ in theta_grid] for t in siou_thrs})

    # bins de tamaño
    sizewise = {name: {("IoU",0.5):dict(tp=0,fp=0,fn=0), ("S-IoU",0.5):dict(tp=0,fp=0,fn=0)} for name,_,_ in SIZE_BINS}
    sizewise["All"] = {("IoU",0.5):dict(tp=0,fp=0,fn=0), ("S-IoU",0.5):dict(tp=0,fp=0,fn=0)}

    pix_agg = dict(pACC=[], mIoU=[], fwIoU=[])

    for im in tqdm(images, desc=f"Eval {model_tag}"):
        fpath = os.path.join(img_dir, im["file_name"])
        img = cv2.imread(fpath)
        if img is None:
            continue
        H,W = img.shape[:2]
        gsd = image_gsd_m(fpath, fallback=0.13)
        gt_masks = ann_masks_for_img(coco, im["id"], H, W)

        # Probabilidades
        probs = tta_probs(predictor, img)
        prob  = probs[1]

        # SemSeg → instancias base (sin filtrar score)
        min_area_m2 = float(min_area_px) * (gsd**2)
        pred_masks_all, pred_scores_all, pred_bin = semseg_to_instances_with_morph(
            prob, thr_components, gsd,
            min_area_m2=min_area_m2,
            morph_close_ks=morph_close_ks, morph_open_ks=morph_open_ks,
            min_score=None
        )

        # Guardar binaria (opcional)
        if write_pred_masks_dir:
            with rasterio.open(fpath) as src:
                meta = src.meta.copy(); meta.update(count=1, dtype=rasterio.uint8, nodata=0)
            os.makedirs(write_pred_masks_dir, exist_ok=True)
            out_path = os.path.join(write_pred_masks_dir, os.path.basename(fpath).replace("Img_","Mask_"))
            with rasterio.open(out_path, "w", **meta) as dst:
                dst.write(pred_bin.astype(np.uint8), 1)

        # Métricas por píxel
        gt_bin = np.any(np.stack(gt_masks,0),0) if len(gt_masks)>0 else np.zeros((H,W), bool)
        pm = pixel_metrics(pred_bin, gt_bin)
        for k,v in pm.items():
            pix_agg[k].append(v)

        # Globales a τ fijo (+ min_score_base, si se usa)
        sel = (lambda s: True) if (min_score_base is None) else (lambda s: s >= float(min_score_base))
        pm_sel = [m for m,s in zip(pred_masks_all, pred_scores_all) if sel(s)]
        ps_sel = [s for s in pred_scores_all if sel(s)]

        for t in iou_thrs:
            m = eval_iou_at_threshold_hungarian(pm_sel, gt_masks, iou_thr=t)
            for k in ("tp","fp","fn"):
                res_counts[("IoU",t)][k] += m[k]
        for t in siou_thrs:
            m = eval_siou_at_threshold(pm_sel, ps_sel, gt_masks, siou_thr=t)
            for k in ("tp","fp","fn"):
                res_counts[("S-IoU",t)][k] += m[k]

        # Tamaños (IoU@0.5 y S-IoU@0.5)
        iou_mat = iou_matrix(pred_masks_all, gt_masks) if (len(pred_masks_all)>0 and len(gt_masks)>0) else np.zeros((len(pred_masks_all), len(gt_masks)))
        assigned=set(); gt_areas = [float((g.sum())*(gsd**2)) for g in gt_masks] if len(gt_masks)>0 else []
        if iou_mat.size>0:
            order = np.argsort(-iou_mat.max(axis=1))
            for i in order:
                if iou_mat.shape[1]==0: break
                j = int(np.argmax(iou_mat[i]))
                if iou_mat[i,j] >= 0.5 and j not in assigned:
                    assigned.add(j)
                    sname = size_label(gt_areas[j])
                    sizewise[sname][("IoU",0.5)]["tp"] += 1; sizewise["All"][("IoU",0.5)]["tp"] += 1
        for j in range(len(gt_masks)):
            if j not in assigned:
                sname = size_label(gt_areas[j]); sizewise[sname][("IoU",0.5)]["fn"] += 1; sizewise["All"][("IoU",0.5)]["fn"] += 1
        for i in range(len(pred_masks_all)):
            if iou_mat.shape[1]==0 or iou_mat[i].max() < 0.5:
                sname = size_label(float(pred_masks_all[i].sum())*(gsd**2))
                sizewise[sname][("IoU",0.5)]["fp"] += 1; sizewise["All"][("IoU",0.5)]["fp"] += 1

        SIOU_BIN_THR = 0.5
        for p in pred_masks_all:
            overlaps = [g for g in gt_masks if np.any(p & g)]
            if not overlaps: continue
            union_gt = np.any(np.stack(overlaps, axis=0), axis=0)
            union_area_m2 = float(union_gt.sum())*(gsd**2)
            bname = size_label(union_area_m2)
            s_pred = siou_pred(p, gt_masks)
            if s_pred >= SIOU_BIN_THR:
                sizewise[bname][("S-IoU",0.5)]["tp"] += 1
                sizewise["All"][("S-IoU",0.5)]["tp"] += 1
            else:
                sizewise[bname][("S-IoU",0.5)]["fp"] += 1
                sizewise["All"][("S-IoU",0.5)]["fp"] += 1
        for g in gt_masks:
            s_lab = siou_label(g, pred_masks_all)
            if s_lab < SIOU_BIN_THR:
                g_area_m2 = float(g.sum())*(gsd**2)
                bname = size_label(g_area_m2)
                sizewise[bname][("S-IoU",0.5)]["fn"] += 1
                sizewise["All"][("S-IoU",0.5)]["fn"] += 1

        # Curvas F1 vs θ_score (τ fijo)
        for tidx, theta in enumerate(theta_grid):
            filt = lambda s: (s >= float(theta))
            pm_t = [m for m,s in zip(pred_masks_all, pred_scores_all) if filt(s)]
            ps_t = [s for s in pred_scores_all if filt(s)]
            for t in iou_thrs:
                m = eval_iou_at_threshold_hungarian(pm_t, gt_masks, iou_thr=t)
                for k in ("tp","fp","fn"):
                    curves_counts[("IoU",t)][tidx][k] += m[k]
            for t in siou_thrs:
                m = eval_siou_at_threshold(pm_t, ps_t, gt_masks, siou_thr=t)
                for k in ("tp","fp","fn"):
                    curves_counts[("S-IoU",t)][tidx][k] += m[k]

    def counts_to_metrics(d):
        tp,fp,fn = d["tp"], d["fp"], d["fn"]
        prec = tp/(tp+fp) if (tp+fp)>0 else 0.0
        rec  = tp/(tp+fn) if (tp+fn)>0 else 0.0
        f1   = 2*prec*rec/(prec+rec) if (prec+rec)>0 else 0.0
        return dict(tp=tp,fp=fp,fn=fn,precision=prec,recall=rec,f1=f1)

    metrics    = {k:counts_to_metrics(v) for k,v in res_counts.items()}
    f1_curves  = {k: [counts_to_metrics(d)["f1"] for d in lst] for k,lst in curves_counts.items()}
    pix_summary= {k: float(np.mean(v)) if len(v)>0 else 0.0 for k,v in pix_agg.items()}
    size_metrics = {sname:{key:counts_to_metrics(cnt) for key,cnt in sub.items()} for sname,sub in sizewise.items()}
    return dict(metrics=metrics, f1_curves=f1_curves, thr_grid=list(map(float, theta_grid)), pixel_metrics=pix_summary, size_metrics=size_metrics)

def to_df_global(res, data_tag, theta):
    rows = []
    for (metric, thr), d in res["metrics"].items():
        rows.append(dict(
            Data=f"{data_tag} (θ={theta:.2f})",
            Metric=metric, Thr=thr,
            TP=d["tp"], FP=d["fp"], FN=d["fn"],
            Precision=100*d["precision"], Recall=100*d["recall"], F1_score=100*d["f1"]
        ))
    return pd.DataFrame(rows)

def to_df_sizes(res, data_tag):
    def f1(p,r): return 100*(2*p*r/max(1e-9,(p+r)) if (p+r)>0 else 0.0)
    rows=[]
    for s in [b[0] for b in SIZE_BINS] + ["All"]:
        ri = res["size_metrics"][s][("IoU",0.5)]; rs = res["size_metrics"][s][("S-IoU",0.5)]
        rows.append({"Data":data_tag,"Size":s,
                     "IoU_P":100*ri["precision"], "IoU_R":100*ri["recall"], "IoU_F1":f1(ri["precision"],ri["recall"]),
                     "S-IoU_P":100*rs["precision"], "S-IoU_R":100*rs["recall"], "S-IoU_F1":f1(rs["precision"],rs["recall"])})
    return pd.DataFrame(rows)

def pick_best_theta(res, key=("S-IoU",0.5)):
    grid = np.array(res["thr_grid"])
    f1   = np.array(res["f1_curves"][key])
    i = int(np.argmax(f1))
    return float(grid[i]), float(f1[i])

def save_curves(res, tag, out_dir, theta_mark=None):
    """ Guarda CSV con las 4 curvas (IoU@0.5/0.75 y S-IoU@0.5/0.75) y dibuja figura. """
    grid = np.array(res["thr_grid"])
    df = pd.DataFrame({
        "theta": grid,
        "F1_IoU_0.5":  100*np.array(res["f1_curves"][("IoU",0.5)]),
        "F1_IoU_0.75": 100*np.array(res["f1_curves"][("IoU",0.75)]),
        "F1_SIoU_0.5": 100*np.array(res["f1_curves"][("S-IoU",0.5)]),
        "F1_SIoU_0.75":100*np.array(res["f1_curves"][("S-IoU",0.75)]),
    })
    os.makedirs(out_dir, exist_ok=True)
    csv_path = os.path.join(out_dir, f"curves_F1_vs_theta_{tag}.csv")
    df.to_csv(csv_path, index=False)

    import matplotlib.pyplot as plt
    plt.rcParams.update({ "font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11, "legend.fontsize": 10, "figure.dpi": 150 })
    fig, ax = plt.subplots(figsize=(6.5, 4.2))
    ax.plot(df["theta"], df["F1_IoU_0.5"],  marker="o", label="IoU @ 0.5")
    ax.plot(df["theta"], df["F1_IoU_0.75"], marker="s", label="IoU @ 0.75")
    ax.plot(df["theta"], df["F1_SIoU_0.5"], marker="^", label="S-IoU @ 0.5")
    ax.plot(df["theta"], df["F1_SIoU_0.75"],marker="D", label="S-IoU @ 0.75")
    if theta_mark is not None:
        ax.axvline(float(theta_mark), linestyle="--", linewidth=1)
    ax.set_xlabel("θ_score"); ax.set_ylabel("F1-score (%)")
    ax.set_title(f"{tag.replace('_',' ')} — F1 vs θ_score (IoU y S-IoU)")
    ax.grid(True, alpha=0.3); ax.legend(loc="best", frameon=False)
    os.makedirs(f"{out_dir}/plots", exist_ok=True)
    png_path = os.path.join(out_dir, f"plots/F1_vs_theta_{tag}_4curves.png")
    plt.savefig(png_path, bbox_inches="tight"); plt.close()
    return csv_path, png_path

def save_masks_at_theta_star(coco_json, img_dir, predictor, inst_params, theta_star, out_dir):
    import rasterio
    coco = COCO(coco_json); images = coco.loadImgs(coco.getImgIds())
    thr_components = float(inst_params["th"])
    morph_open_ks  = int(inst_params.get("morph_open_ks", 0))
    morph_close_ks = int(inst_params.get("morph_close_ks", 0))
    min_area_px    = int(inst_params.get("min_area_px", 0))
    theta_star     = float(theta_star)
    os.makedirs(out_dir, exist_ok=True)
    for im in tqdm(images, desc=f"Write θ*={theta_star:.2f}"):
        fpath = os.path.join(img_dir, im["file_name"])
        img = cv2.imread(fpath)
        if img is None:
            continue
        H, W = img.shape[:2]
        gsd = image_gsd_m(fpath, fallback=0.13)
        # Probabilidades → prob clase 1
        probs = tta_probs(predictor, img)
        prob  = probs[1]
        # instancias base
        min_area_m2 = float(min_area_px) * (gsd**2)
        pred_masks_all, pred_scores_all, _ = semseg_to_instances_with_morph(
            prob, thr_components, gsd,
            min_area_m2=min_area_m2,
            morph_close_ks=morph_close_ks, morph_open_ks=morph_open_ks,
            min_score=None
        )
        # reconstruir binaria con score ≥ θ*
        pred_bin_sel = np.zeros((H, W), dtype=np.uint8)
        for m, s in zip(pred_masks_all, pred_scores_all):
            if s >= theta_star:
                pred_bin_sel[m] = 1
        with rasterio.open(fpath) as src:
            meta = src.meta.copy(); meta.update(count=1, dtype=rasterio.uint8, nodata=0)
        out_name = os.path.basename(fpath).replace("Img_", "Mask_")
        out_path = os.path.join(out_dir, out_name)
        with rasterio.open(out_path, "w", **meta) as dst:
            dst.write(pred_bin_sel, 1)

In [ ]:
#@title 7. Curvas/θ*, métricas globales/tamaños, máscaras θ* y métricas pixelares (DeepLabV3+)
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display

# Helper: guardar CSV + figura con 4 curvas y marcar θ*
def _save_curves_with_mark(res, tag, out_dir, theta_mark=None):
    grid = np.array(res["thr_grid"])
    df = pd.DataFrame({
        "theta": grid,
        "F1_IoU_0.5":  100*np.array(res["f1_curves"][("IoU",0.5)]),
        "F1_IoU_0.75": 100*np.array(res["f1_curves"][("IoU",0.75)]),
        "F1_SIoU_0.5": 100*np.array(res["f1_curves"][("S-IoU",0.5)]),
        "F1_SIoU_0.75":100*np.array(res["f1_curves"][("S-IoU",0.75)]),
    })
    os.makedirs(out_dir, exist_ok=True)
    csv_path = os.path.join(out_dir, f"curves_F1_vs_theta_{tag}.csv")
    df.to_csv(csv_path, index=False)
    plt.rcParams.update({ "font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11, "legend.fontsize": 10, "figure.dpi": 150 })
    fig, ax = plt.subplots(figsize=(6.5, 4.2))
    ax.plot(df["theta"], df["F1_IoU_0.5"],  marker="o", label="IoU @ 0.5")
    ax.plot(df["theta"], df["F1_IoU_0.75"], marker="s", label="IoU @ 0.75")
    ax.plot(df["theta"], df["F1_SIoU_0.5"], marker="^", label="S-IoU @ 0.5")
    ax.plot(df["theta"], df["F1_SIoU_0.75"],marker="D", label="S-IoU @ 0.75")
    if theta_mark is not None:
        ax.axvline(float(theta_mark), linestyle="--", linewidth=1)
    ax.set_xlabel("θ_score"); ax.set_ylabel("F1-score (%)")
    ax.set_title(f"{tag} — F1 vs θ_score (IoU y S-IoU) — DeepLab")
    ax.grid(True, alpha=0.3); ax.legend(loc="best", frameon=False)
    os.makedirs(f"{out_dir}/plots", exist_ok=True)
    png_path = os.path.join(out_dir, f"plots/F1_vs_theta_{tag}_4curves.png")
    plt.savefig(png_path, bbox_inches="tight"); plt.close()
    return csv_path, png_path

# 1) Curvas F1 vs θ y θ* (sin forzar valores)
inst_params_PI = {"th": 0.50, "min_area_px": 0, "morph_open_ks": 0, "morph_close_ks": 0, "min_score": None}
inst_params_FW = {"th": 0.50, "min_area_px": 0, "morph_open_ks": 0, "morph_close_ks": 0, "min_score": None}

print("PI (curvas vs θ_score)…")
res_pi = evaluate_semseg_vs_coco_params_hungarian(
    PI_COCO_JSON, PI_IMG_DIR, predictor_pi, inst_params_PI,
    write_pred_masks_dir=PRED_PI, model_tag="DeepLab-PI",
    base_pixel_thr=inst_params_PI["th"], theta_grid=np.linspace(0.05, 0.95, 19)
)
theta_pi, f1_pi = pick_best_theta(res_pi, ("S-IoU",0.5))
csv_pi, png_pi  = _save_curves_with_mark(res_pi, "PI", OUT_PI, theta_mark=theta_pi)
print(f"PI: θ* (S-IoU@0.5) = {theta_pi:.2f} | F1 = {100*f1_pi:.2f}%")
print("Curvas PI guardadas en:", csv_pi, "y", png_pi)

print("\nFW (curvas vs θ_score)…")
res_fw = evaluate_semseg_vs_coco_params_hungarian(
    FW_COCO_JSON, FW_IMG_DIR, predictor_fw, inst_params_FW,
    write_pred_masks_dir=PRED_FW, model_tag="DeepLab-FW",
    base_pixel_thr=inst_params_FW["th"], theta_grid=np.linspace(0.05, 0.95, 19)
)
theta_fw, f1_fw = pick_best_theta(res_fw, ("S-IoU",0.5))
csv_fw, png_fw  = _save_curves_with_mark(res_fw, "FW", OUT_PI, theta_mark=theta_fw)
print(f"FW: θ* (S-IoU@0.5) = {theta_fw:.2f} | F1 = {100*f1_fw:.2f}%")
print("Curvas FW guardadas en:", csv_fw, "y", png_fw)

# 2) Métricas globales/tamaños a θ*
def metrics_at_theta_star(coco_json, img_dir, predictor, inst_params, theta_star, tag_write_dir=None):
    pars = dict(inst_params)
    pars["min_score"] = float(theta_star)  # filtra instancias por score ≥ θ*
    return evaluate_semseg_vs_coco_params_hungarian(
        coco_json, img_dir, predictor, pars,
        write_pred_masks_dir=tag_write_dir, model_tag=f"DeepLab@θ={theta_star:.2f}",
        base_pixel_thr=inst_params["th"], theta_grid=np.linspace(0.05, 0.95, 19)
    )

print("\nEvaluando a θ* (global y tamaños)…")
res_pi_star = metrics_at_theta_star(PI_COCO_JSON, PI_IMG_DIR, predictor_pi, inst_params_PI, theta_pi, tag_write_dir=PRED_PI)
res_fw_star = metrics_at_theta_star(FW_COCO_JSON, FW_IMG_DIR, predictor_fw, inst_params_FW, theta_fw, tag_write_dir=PRED_FW)

df_pi = to_df_global(res_pi_star, "PI-test", theta_pi)
df_fw = to_df_global(res_fw_star, "FW-test", theta_fw)
df_all = pd.concat([df_pi, df_fw], ignore_index=True)
from IPython.display import display
display(df_all)

df_pi_sz = to_df_sizes(res_pi_star, "PI-test")
df_fw_sz = to_df_sizes(res_fw_star, "FW-test")
display(df_pi_sz); display(df_fw_sz)

# Guardar tablas/CSV
os.makedirs(f"{OUT_PI}/csv", exist_ok=True)
os.makedirs(f"{OUT_PI}/tables", exist_ok=True)
df_all.to_csv(f"{OUT_PI}/csv/global_metrics_DeepLab_theta_star.csv", index=False)
with open(f"{OUT_PI}/tables/table_global_DeepLab_theta_star.tex","w") as f:
    f.write(df_all.to_latex(index=False, float_format='%.4f'))
df_pi_sz.to_csv(f"{OUT_PI}/csv/sizewise_PI_DeepLab_theta_star.csv", index=False)
df_fw_sz.to_csv(f"{OUT_PI}/csv/sizewise_FW_DeepLab_theta_star.csv", index=False)
with open(f"{OUT_PI}/tables/table_sizewise_PI_DeepLab_theta_star.tex","w") as f:
    f.write(df_pi_sz.to_latex(index=False, float_format='%.2f'))
with open(f"{OUT_PI}/tables/table_sizewise_FW_DeepLab_theta_star.tex","w") as f:
    f.write(df_fw_sz.to_latex(index=False, float_format='%.2f'))
print("\n Métricas globales/tamaños guardadas.")

# 3) Guardar máscaras binarias a θ* (para métricas pixelares / WS)
PRED_PI_TH = f"{OUT_PI}/Predictions_theta_star_DeepLab"
PRED_FW_TH = f"{OUT_FW}/Predictions_theta_star_DeepLab"
os.makedirs(PRED_PI_TH, exist_ok=True)
os.makedirs(PRED_FW_TH, exist_ok=True)
save_masks_at_theta_star(PI_COCO_JSON, PI_IMG_DIR, predictor_pi, inst_params_PI, theta_pi, PRED_PI_TH)
save_masks_at_theta_star(FW_COCO_JSON, FW_IMG_DIR, predictor_fw, inst_params_FW, theta_fw, PRED_FW_TH)
print("Máscaras a θ* guardadas en:")
print(" ", PRED_PI_TH)
print(" ", PRED_FW_TH)

# 4) Métricas pixelares a θ*
def read_mask(path):
    with rasterio.open(path) as src:
        return src.read(1)

def confusion_from_masks(gt, pr, num_classes=2):
    cm = np.zeros((num_classes, num_classes), dtype=np.int64)
    gt = gt.flatten(); pr = pr.flatten()
    valid = (gt >= 0) & (gt < num_classes)
    gt = gt[valid]; pr = pr[valid]
    for i in range(num_classes):
        for j in range(num_classes):
            cm[i, j] += np.sum((gt == i) & (pr == j))
    return cm

def miou_pixacc_fwIoU_folder(gt_dir, pred_dir, num_classes=2):
    files = sorted([f for f in os.listdir(gt_dir) if f.endswith(".tif")])
    cm_total = np.zeros((num_classes, num_classes), dtype=np.int64)
    for f in files:
        gt_path = os.path.join(gt_dir, f)
        pr_path = os.path.join(pred_dir, f)
        if not os.path.exists(pr_path):
            alt = f.replace("Img_", "Mask_")
            pr_path = os.path.join(pred_dir, alt)
            if not os.path.exists(pr_path):
                continue
        gt = read_mask(gt_path); pr = read_mask(pr_path)
        cm_total += confusion_from_masks(gt, pr, num_classes)
    tp = np.diag(cm_total); total = cm_total.sum()
    pixacc = float(tp.sum() / total) if total>0 else 0.0
    ious = []
    for c in range(num_classes):
        denom = tp[c] + (cm_total[c,:].sum()-tp[c]) + (cm_total[:,c].sum()-tp[c])
        ious.append(float(tp[c]/denom) if denom>0 else 0.0)
    miou = float(np.mean(ious)) if ious else 0.0
    freq = cm_total.sum(axis=1) / total if total>0 else np.zeros(num_classes)
    fwiou = float((freq * np.array(ious)).sum())
    return dict(mIoU=miou, pixAcc=pixacc, fwIoU=fwiou, IoUs=ious, CM=cm_total)

pix_PI = miou_pixacc_fwIoU_folder(GT_PI, PRED_PI_TH, num_classes=2)
pix_FW = miou_pixacc_fwIoU_folder(GT_FW, PRED_FW_TH, num_classes=2)
df_pix = pd.DataFrame([
    dict(Split="PI-test", mIoU=pix_PI["mIoU"], pixAcc=pix_PI["pixAcc"], fwIoU=pix_PI["fwIoU"]),
    dict(Split="FW-test", mIoU=pix_FW["mIoU"], pixAcc=pix_FW["pixAcc"], fwIoU=pix_FW["fwIoU"]),
])
display(df_pix)
df_pix.to_csv(f"{OUT_PI}/csv/pixel_metrics_DeepLab_theta_star.csv", index=False)
with open(f"{OUT_PI}/tables/table_pixel_metrics_DeepLab_theta_star.tex","w") as f:
    f.write(df_pix.to_latex(index=False, float_format="%.4f"))
print("\n Métricas pixelares guardadas.")
print("\n Figuras de curvas estilo paper generadas en:")
print(" -", os.path.join(OUT_PI, "plots", "F1_vs_theta_PI_DeepLab_4curves.png"))
print(" -", os.path.join(OUT_PI, "plots", "F1_vs_theta_FW_DeepLab_4curves.png"))

In [ ]:
#@title 8. Cobertura/Densidad a θ* + WS baseline y calibrado (FW)
import os, math
import numpy as np
import pandas as pd
import rasterio
from scipy import ndimage
from sklearn.metrics import mean_squared_error, r2_score

# --- Utils mínimos ---
def read_mask(path):
    with rasterio.open(path) as src:
        return src.read(1)

# WS helpers (skimage opcional; fallback a CC)
try:
    from skimage.feature import peak_local_max
    from skimage.segmentation import watershed
    from skimage.morphology import h_minima
    _SKIMAGE_OK = True
except Exception:
    _SKIMAGE_OK = False
    peak_local_max = watershed = h_minima = None
    print("scikit-image no disponible: usaré componentes conexas como fallback.")

def split_ws_pred(mask_bin, min_dist_px=3, h_rel=0.10):
    if not _SKIMAGE_OK:
        labels, _ = ndimage.label(mask_bin.astype(np.uint8))
        return labels
    from scipy import ndimage as ndi
    mask_bin = mask_bin.astype(np.uint8)
    dist = ndi.distance_transform_edt(mask_bin)
    if dist.max() > 0 and h_rel > 0:
        try:
            dist_supp = dist - h_minima(dist, h=float(h_rel * dist.max()))
        except Exception:
            dist_supp = dist
    else:
        dist_supp = dist
    coords = peak_local_max(dist_supp, min_distance=int(max(1, min_dist_px)), labels=mask_bin)
    markers = np.zeros_like(mask_bin, dtype=np.int32)
    for i, (r, c) in enumerate(coords, start=1):
        markers[r, c] = i
    labels = watershed(-dist_supp, markers, mask=mask_bin)
    return labels

def density_cc(mask_bin):
    _, n = ndimage.label(mask_bin.astype(np.uint8))
    return int(n)

def density_ws(mask_bin, min_dist_px=3, h_rel=0.10):
    labels = split_ws_pred(mask_bin.astype(np.uint8), min_dist_px=min_dist_px, h_rel=h_rel)
    return int(labels.max())

# -------- Cobertura y densidad por imagen (con NoData enmascarado) --------
def coverage_density_from_folders(gt_dir, pr_dir, ws_params=None):
    """
    Calcula cobertura (fracción 0–1) y densidad (conteo por imagen) para cada GT/Pred.
    - Cobertura SOLO sobre píxeles válidos (NoData del GT).
    - Devuelve métricas de cobertura en **%** (RMSE/MAE/MBE) y R².
    - Devuelve métricas de densidad en unidades absolutas y R².
    - 'series' mantiene cov_T/cov_P en fracción (0–1) para figuras.
    """
    files = sorted([f for f in os.listdir(gt_dir) if f.endswith(".tif")])
    cov_T, cov_P, den_T, den_P = [], [], [], []

    for f in files:
        gt_path = os.path.join(gt_dir, f)
        pr_path = os.path.join(pr_dir, f)
        if not os.path.exists(pr_path):
            alt = f.replace("Img_", "Mask_")
            pr_path = os.path.join(pr_dir, alt)
            if not os.path.exists(pr_path):
                continue

        gt = read_mask(gt_path); pr = read_mask(pr_path)

        # Enmascarado NoData desde GT
        with rasterio.open(gt_path) as src:
            nod = src.nodata
        valid = (gt != nod) if nod is not None else np.ones_like(gt, dtype=bool)
        if valid.sum() == 0:
            continue

        gtb = (gt == 1) & valid
        prb = (pr == 1) & valid

        # Cobertura (0–1) sobre válidos
        cov_T.append(float(gtb.sum() / valid.sum()))
        cov_P.append(float(prb.sum() / valid.sum()))

        # Densidad (conteo de objetos)
        den_T.append(density_cc(gtb))
        if ws_params is None:
            den_P.append(density_cc(prb))
        else:
            den_P.append(density_ws(prb,
                                    min_dist_px=int(ws_params.get("min_dist_px", 3)),
                                    h_rel=float(ws_params.get("h_rel", 0.10))))

    # Cobertura en % (puntos porcentuales) + R²
    if cov_T:
        rmse_cover = float(np.sqrt(mean_squared_error(cov_T, cov_P))) * 100.0
        mae_cover  = float(np.mean(np.abs(np.array(cov_T) - np.array(cov_P)))) * 100.0
        mbe_cover  = float(np.mean(np.array(cov_P) - np.array(cov_T))) * 100.0
        r2_cover   = float(r2_score(cov_T, cov_P))
    else:
        rmse_cover = mae_cover = mbe_cover = r2_cover = float("nan")

    # Densidad en unidades absolutas + R²
    if den_T:
        rmse_density = float(np.sqrt(mean_squared_error(den_T, den_P)))
        mae_density  = float(np.mean(np.abs(np.array(den_T) - np.array(den_P))))
        mbe_density  = float(np.mean(np.array(den_P) - np.array(den_T)))
        r2_density   = float(r2_score(den_T, den_P))
    else:
        rmse_density = mae_density = mbe_density = r2_density = float("nan")

    out = dict(
        N=len(cov_T),
        RMSE_cover=rmse_cover, MAE_cover=mae_cover, MBE_cover=mbe_cover, R2_cover=r2_cover,
        RMSE_density=rmse_density, MAE_density=mae_density, MBE_density=mbe_density, R2_density=r2_density
    )
    return out, dict(cov_T=cov_T, cov_P=cov_P, den_T=den_T, den_P=den_P)

# -------- Área válida (ha) y densidad por hectárea --------
try:
    from pyproj import Geod
    _HAS_PYPROJ = True
    _GEOD = Geod(ellps="WGS84")
except Exception:
    _HAS_PYPROJ = False
    _GEOD = None

def _poly_area_m2_from_bounds(l, b, r, t):
    if not _HAS_PYPROJ:
        return None
    lons = [l, l, r, r, l]; lats = [b, t, t, b, b]
    area, _ = _GEOD.polygon_area_perimeter(lons, lats)
    return abs(area)

def _m_per_deg_lat(lat_rad):
    return (111132.92 - 559.82*math.cos(2*lat_rad) + 1.175*math.cos(4*lat_rad) - 0.0023*math.cos(6*lat_rad))

def _m_per_deg_lon(lat_rad):
    return (111412.84*math.cos(lat_rad) - 93.5*math.cos(3*lat_rad) + 0.118*math.cos(5*lat_rad))

def raster_valid_area_ha(img_path, valid_mask=None):
    with rasterio.open(img_path) as src:
        H, W = src.height, src.width
        tr, crs, bounds = src.transform, src.crs, src.bounds
        nodata = src.nodata
        if valid_mask is None:
            try:
                arr = src.read(1)
                valid_mask = (arr != nodata) if nodata is not None else np.ones((H, W), dtype=bool)
            except Exception:
                valid_mask = np.ones((H, W), dtype=bool)
        valid_px = int(np.sum(valid_mask))
        if valid_px == 0:
            return 0.0

        det = abs(tr.a * tr.e - tr.b * tr.d)
        if crs is not None and getattr(crs, "is_projected", False):
            return (det * valid_px) / 10000.0

        area_geo_m2 = _poly_area_m2_from_bounds(bounds.left, bounds.bottom, bounds.right, bounds.top)
        if area_geo_m2 is not None:
            frac = valid_px / float(H * W)
            return (area_geo_m2 * frac) / 10000.0

        lat_c = 0.5 * (bounds.bottom + bounds.top)
        lat_rad = math.radians(lat_c)
        mx = _m_per_deg_lon(lat_rad); my = _m_per_deg_lat(lat_rad)
        px_deg_x = tr.a; px_deg_y = abs(tr.e)
        px_m2 = (mx * px_deg_x) * (my * px_deg_y)
        return (valid_px * px_m2) / 10000.0

def eval_density_per_ha(gt_dir, pr_dir, ws_params=None):
    """
    Densidad (ind/ha), usando área válida (NoData enmascarado) y WS opcional.
    Devuelve y_true/y_pred + RMSE y R² en ind/ha.
    """
    files = sorted([f for f in os.listdir(gt_dir) if f.endswith(".tif")])
    yT, yP = [], []
    for f in files:
        gt_path = os.path.join(gt_dir, f)
        pr_path = os.path.join(pr_dir, f)
        if not os.path.exists(pr_path):
            alt = f.replace("Img_", "Mask_")
            pr_path = os.path.join(pr_dir, alt)
            if not os.path.exists(pr_path):
                continue

        gt = read_mask(gt_path); pr = read_mask(pr_path)
        with rasterio.open(gt_path) as src:
            nod = src.nodata
        valid = (gt != nod) if nod is not None else np.ones_like(gt, bool)
        area_ha = max(raster_valid_area_ha(gt_path, valid_mask=valid), 1e-9)

        gtb = (gt == 1) & valid
        prb = (pr == 1) & valid

        labs_gt, _ = ndimage.label(gtb.astype(np.uint8))
        dens_gt = int(labs_gt.max()) / area_ha

        if ws_params is None:
            labs_pr, _ = ndimage.label(prb.astype(np.uint8))
            dens_pr = int(labs_pr.max()) / area_ha
        else:
            labs_pr = split_ws_pred(prb.astype(np.uint8),
                                    min_dist_px=int(ws_params.get("min_dist_px", 3)),
                                    h_rel=float(ws_params.get("h_rel", 0.10)))
            dens_pr = int(labs_pr.max()) / area_ha

        yT.append(dens_gt); yP.append(dens_pr)

    rmse = float(np.sqrt(mean_squared_error(yT, yP))) if yT else np.nan
    r2   = float(r2_score(yT, yP)) if yT else np.nan
    return dict(y_true=yT, y_pred=yP, rmse=rmse, r2=r2)

# -------- Calibración WS por imagen (minimiza RMSE de conteo) --------
def calibrate_ws_density(gt_dir, pr_dir,
                         grid_min_dist=(2,3,4,5,6,7),
                         grid_h=(0.05,0.08,0.10,0.12,0.15,0.20)):
    files = sorted([f for f in os.listdir(gt_dir) if f.endswith(".tif")])
    best = {"rmse": 1e9, "min_dist_px": None, "h_rel": None}
    for dmin in grid_min_dist:
        for h in grid_h:
            yT, yP = [], []
            for f in files:
                gt_path = os.path.join(gt_dir, f)
                pr_path = os.path.join(pr_dir, f)
                if not os.path.exists(pr_path):
                    alt = f.replace("Img_", "Mask_")
                    pr_path = os.path.join(pr_dir, alt)
                    if not os.path.exists(pr_path):
                        continue
                gt = read_mask(gt_path); pr = read_mask(pr_path)
                gtb = (gt == 1); prb = (pr == 1)
                labs_gt,_ = ndimage.label(gtb.astype(np.uint8)); dens_gt = int(labs_gt.max())
                labs_pr = split_ws_pred(prb.astype(np.uint8), min_dist_px=int(dmin), h_rel=float(h))
                dens_pr = int(labs_pr.max())
                yT.append(dens_gt); yP.append(dens_pr)
            if yT:
                rmse = float(np.sqrt(mean_squared_error(yT, yP)))
                if rmse < best["rmse"]:
                    best = {"rmse": rmse, "min_dist_px": int(dmin), "h_rel": float(h)}
    return best

# ================== Baseline + Calibración (usa tus rutas GT_* y PRED_* existentes) ==================
WS_PARAMS = dict(min_dist_px=3, h_rel=0.10)

# 1) Métricas por imagen (baseline)
pi_cd, _ = coverage_density_from_folders(GT_PI, PRED_PI_TH, ws_params=WS_PARAMS)
fw_cd, _ = coverage_density_from_folders(GT_FW, PRED_FW_TH, ws_params=WS_PARAMS)
print("\nCobertura/Densidad por imagen (θ* | WS baseline):")
print("PI: RMSE_cover={:.2f}%  MAE_cover={:.2f}%  MBE_cover={:+.2f}%  |  R²_cover={:.4f}  ||  RMSE_dens={:.2f}  MAE_dens={:.2f}  MBE_dens={:+.2f}  R²_dens={:.3f}".format(
    pi_cd["RMSE_cover"], pi_cd["MAE_cover"], pi_cd["MBE_cover"], pi_cd["R2_cover"], pi_cd["RMSE_density"], pi_cd["MAE_density"], pi_cd["MBE_density"], pi_cd["R2_density"]))
print("FW: RMSE_cover={:.2f}%  MAE_cover={:.2f}%  MBE_cover={:+.2f}%  |  R²_cover={:.4f}  ||  RMSE_dens={:.2f}  MAE_dens={:.2f}  MBE_dens={:+.2f}  R²_dens={:.3f}".format(
    fw_cd["RMSE_cover"], fw_cd["MAE_cover"], fw_cd["MBE_cover"], fw_cd["R2_cover"], fw_cd["RMSE_density"], fw_cd["MAE_density"], fw_cd["MBE_density"], fw_cd["R2_density"]))

# 2) Densidad por hectárea (baseline)
pi_ha = eval_density_per_ha(GT_PI, PRED_PI_TH, ws_params=WS_PARAMS)
fw_ha = eval_density_per_ha(GT_FW, PRED_FW_TH, ws_params=WS_PARAMS)
print("\nDensidad por hectárea (θ* | WS baseline):")
print("PI: n={} | RMSE={:.2f} ind/ha | R²={:.3f}".format(len(pi_ha["y_true"]), pi_ha["rmse"], pi_ha["r2"]))
print("FW: n={} | RMSE={:.2f} ind/ha | R²={:.3f}".format(len(fw_ha["y_true"]), fw_ha["rmse"], fw_ha["r2"]))

# 3) Calibración WS en FW (usa PRED_FW_TH si existen binarios a θ*)
_FW_PRED_DIR_FOR_WS = PRED_FW_TH if (os.path.isdir(PRED_FW_TH) and len([f for f in os.listdir(PRED_FW_TH) if f.endswith(".tif")])>0) else PRED_FW
best_ws_fw = calibrate_ws_density(GT_FW, _FW_PRED_DIR_FOR_WS)
print("\nWS FW óptimo:", best_ws_fw)
WS_PARAMS_FW = dict(min_dist_px=int(best_ws_fw["min_dist_px"]), h_rel=float(best_ws_fw["h_rel"]))

fw_cd_cal, _ = coverage_density_from_folders(GT_FW, _FW_PRED_DIR_FOR_WS, ws_params=WS_PARAMS_FW)
fw_ha_cal = eval_density_per_ha(GT_FW, _FW_PRED_DIR_FOR_WS, ws_params=WS_PARAMS_FW)

print("\nCobertura/Densidad FW (θ* + WS calibrado):")
print("FW: RMSE_cover={:.2f}%  MAE_cover={:.2f}%  MBE_cover={:+.2f}%  |  R²_cover={:.4f}  ||  RMSE_dens={:.2f}  MAE_dens={:.2f}  MBE_dens={:+.2f}  R²_dens={:.3f}".format(
    fw_cd_cal["RMSE_cover"], fw_cd_cal["MAE_cover"], fw_cd_cal["MBE_cover"], fw_cd_cal["R2_cover"],
    fw_cd_cal["RMSE_density"], fw_cd_cal["MAE_density"], fw_cd_cal["MBE_density"], fw_cd_cal["R2_density"]))
print("\nDensidad por hectárea FW (θ* + WS calibrado):")
print("FW: n={} | RMSE={:.2f} ind/ha | R²={:.3f}".format(len(fw_ha_cal["y_true"]), fw_ha_cal["rmse"], fw_ha_cal["r2"]))

# 4) Guardado de resumen WS calibrado (DeepLab)
os.makedirs(f"{OUT_PI}/csv", exist_ok=True)
os.makedirs(f"{OUT_PI}/tables", exist_ok=True)
df_fw_ws = pd.DataFrame([dict(
    Split="FW-test (DeepLabV3+)",
    WS_min_dist_px=WS_PARAMS_FW["min_dist_px"], WS_h_rel=WS_PARAMS_FW["h_rel"],
    RMSE_cover=fw_cd_cal["RMSE_cover"], MAE_cover=fw_cd_cal["MAE_cover"], MBE_cover=fw_cd_cal["MBE_cover"], R2_cover=fw_cd_cal["R2_cover"],
    RMSE_density=fw_cd_cal["RMSE_density"], MAE_density=fw_cd_cal["MAE_density"], MBE_density=fw_cd_cal["MBE_density"], R2_density=fw_cd_cal["R2_density"],
    RMSE_dens_ha=fw_ha_cal["rmse"], R2_dens_ha=fw_ha_cal["r2"]
)])
csv_path = f"{OUT_PI}/csv/fw_ws_calibrated_summary_DeepLab.csv"
tex_path = f"{OUT_PI}/tables/table_fw_ws_calibrated_summary_DeepLab.tex"
df_fw_ws.to_csv(csv_path, index=False)
with open(tex_path, "w") as f:
    f.write(df_fw_ws.to_latex(index=False, float_format="%.4f"))
print("\nGuardado resumen WS calibrado (DeepLab) en:")
print(" CSV  ->", csv_path)
print(" LaTeX->", tex_path)

In [ ]:
#@title 9. Figuras extra (DeepLab): Scatter FW y Barras por tamaño (PI/FW)
import os, numpy as np, matplotlib.pyplot as plt
from scipy.stats import pearsonr

#  Utilidades métricas para anotación en scatter
def _rmse(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2))) if len(y_true) else float("nan")
def _mae(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return float(np.mean(np.abs(y_true - y_pred))) if len(y_true) else float("nan")
def _mbe(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true), np.asarray(y_pred)
    return float(np.mean(y_pred - y_true)) if len(y_true) else float("nan")

os.makedirs(f"{OUT_PI}/plots", exist_ok=True)

#  1) Scatter FW: cobertura (%) y densidad (ind/ha) a θ_fw
try:
    _WS = WS_PARAMS_FW
except NameError:
    try:
        _WS = WS_PARAMS
    except NameError:
        _WS = dict(min_dist_px=3, h_rel=0.10)

# Fuente de máscaras para evaluar: PRED_FW_TH (θ*) si existen, si no PRED_FW
_FW_DIR = PRED_FW_TH if (os.path.isdir(PRED_FW_TH) and len([f for f in os.listdir(PRED_FW_TH) if f.endswith(".tif")])>0) else PRED_FW

_, _series_fw = coverage_density_from_folders(GT_FW, _FW_DIR, ws_params=_WS)
_cov_T = np.array(_series_fw["cov_T"]) * 100.0
_cov_P = np.array(_series_fw["cov_P"]) * 100.0

_den_fw = eval_density_per_ha(GT_FW, _FW_DIR, ws_params=_WS)
_den_T = np.array(_den_fw["y_true"])
_den_P = np.array(_den_fw["y_pred"])

_theta_fw_txt = f"{theta_fw:.2f}" if 'theta_fw' in globals() else "?"
plt.rcParams.update({
    "font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11,
    "legend.fontsize": 10, "figure.dpi": 150
})
fig, axs = plt.subplots(1, 2, figsize=(10, 4))

# (a) Cobertura
r_cov = pearsonr(_cov_T, _cov_P)[0] if len(_cov_T) > 1 else np.nan
rmse_cov, mae_cov, mbe_cov = _rmse(_cov_T, _cov_P), _mae(_cov_T, _cov_P), _mbe(_cov_T, _cov_P)
axs[0].scatter(_cov_T, _cov_P, s=12)
lim = [0, max(1e-6, _cov_T.max(), _cov_P.max()) * 1.05]
axs[0].plot(lim, lim, linestyle="--")
axs[0].set_xlim(lim); axs[0].set_ylim(lim)
axs[0].set_xlabel("Observed canopy cover (%)"); axs[0].set_ylabel("Predicted canopy cover (%)")
axs[0].set_title(f"(a) FW — Cover @ θ={_theta_fw_txt}")
axs[0].text(0.02, 0.98, f"r={r_cov:.2f}\nRMSE={rmse_cov:.2f}%\nMAE={mae_cov:.2f}%\nMBE={mbe_cov:.2f}%",
            transform=axs[0].transAxes, va="top")

# (b) Densidad
r_den = pearsonr(_den_T, _den_P)[0] if len(_den_T) > 1 else np.nan
rmse_den, mae_den, mbe_den = _rmse(_den_T, _den_P), _mae(_den_T, _den_P), _mbe(_den_T, _den_P)
axs[1].scatter(_den_T, _den_P, s=12)
lim2 = [0, max(1e-6, _den_T.max(), _den_P.max()) * 1.05]
axs[1].plot(lim2, lim2, linestyle="--")
axs[1].set_xlim(lim2); axs[1].set_ylim(lim2)
axs[1].set_xlabel("Observed shrubs per ha"); axs[1].set_ylabel("Predicted shrubs per ha")
axs[1].set_title(f"(b) FW — Density @ θ={_theta_fw_txt}")
axs[1].text(0.02, 0.98, f"r={r_den:.2f}\nRMSE={rmse_den:.2f}\nMAE={mae_den:.2f}\nMBE={mbe_den:.2f}",
            transform=axs[1].transAxes, va="top")

fig.suptitle("Observed vs Predicted — FW — DeepLab")
fig.tight_layout(rect=[0, 0, 1, 0.93])
_scatter_path = f"{OUT_PI}/plots/FW_scatter_cover_density_DeepLab.png"
plt.savefig(_scatter_path, bbox_inches="tight")
plt.close()
print(" Scatter FW guardado en:", _scatter_path)

# 2) Barras por tamaño (PI y FW): F1 IoU@0.5 y S-IoU@0.5
_theta_pi = globals().get('theta_pi', 0.5)
_theta_fw = globals().get('theta_fw', 0.5)

def _get_sizewise(res, data_tag):
    def f1(p, r): return 100*(2*p*r / max(1e-9, (p+r)) if (p+r)>0 else 0.0)
    rows=[]
    for s in [b[0] for b in SIZE_BINS] + ["All"]:
        ri = res["size_metrics"][s][("IoU",0.5)]
        rs = res["size_metrics"][s][("S-IoU",0.5)]
        rows.append({"Data":data_tag, "Size":s,
                     "IoU_F1":f1(ri["precision"], ri["recall"]),
                     "S-IoU_F1":f1(rs["precision"], rs["recall"])})
    return pd.DataFrame(rows)

try:
    _df_pi_sz = df_pi_sz.copy()
    if not {"IoU_F1","S-IoU_F1","Size"}.issubset(_df_pi_sz.columns):
        _df_pi_sz = _get_sizewise(res_pi_star, "PI-test")
except NameError:
    _df_pi_sz = _get_sizewise(res_pi_star, "PI-test")

try:
    _df_fw_sz = df_fw_sz.copy()
    if not {"IoU_F1","S-IoU_F1","Size"}.issubset(_df_fw_sz.columns):
        _df_fw_sz = _get_sizewise(res_fw_star, "FW-test")
except NameError:
    _df_fw_sz = _get_sizewise(res_fw_star, "FW-test")

_order = [b[0] for b in SIZE_BINS] + ["All"]
_df_pi_sz = _df_pi_sz.set_index("Size").reindex(_order).reset_index()
_df_fw_sz = _df_fw_sz.set_index("Size").reindex(_order).reset_index()

def _plot_sizewise_bars(df_sz, split_name, out_name, theta_txt):
    if not {"IoU_F1","S-IoU_F1","Size"}.issubset(df_sz.columns):
        raise RuntimeError("El DataFrame de tamaños no contiene columnas esperadas: IoU_F1, S-IoU_F1, Size")
    labels = df_sz["Size"].tolist()
    x = np.arange(len(labels)); width = 0.38
    plt.rcParams.update({
        "font.size": 11, "axes.titlesize": 12, "axes.labelsize": 11,
        "legend.fontsize": 10, "figure.dpi": 150
    })
    fig, ax = plt.subplots(figsize=(9.5, 4.2))
    ax.bar(x - width/2, df_sz["IoU_F1"].values,  width, label="IoU @ 0.5")
    ax.bar(x + width/2, df_sz["S-IoU_F1"].values, width, label="S-IoU @ 0.5")
    ax.set_xticks(x, labels)
    ax.set_ylabel("F1-score (%)"); ax.set_xlabel("Size bin")
    ax.set_title(f"{split_name} — Size-wise F1 (θ={theta_txt}) — DeepLab")
    ax.grid(axis="y", alpha=0.3); ax.legend(loc="best", frameon=False)
    _out = f"{OUT_PI}/plots/{out_name}"
    plt.savefig(_out, bbox_inches="tight"); plt.close()
    return _out

_pi_bars_path = _plot_sizewise_bars(_df_pi_sz, "PI", "PI_sizewise_bars_DeepLab.png", f"{_theta_pi:.2f}")
_fw_bars_path = _plot_sizewise_bars(_df_fw_sz, "FW", "FW_sizewise_bars_DeepLab.png", f"{_theta_fw:.2f}")

print(" Barras por tamaño guardadas en:")
print(" -", _pi_bars_path)
print(" -", _fw_bars_path)